In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are loyal and friendly animals that make great companions.",
        metadata={"source": "pets-doc"}
    ),

    Document(
        page_content="Cats are independent pets that enjoy spending time in their own space.",
        metadata={"source": "pets-doc"}
    ),

    Document(
        page_content="Goldfish are easy-to-care-for pets and are suitable for beginners.",
        metadata={"source": "fish-pets-doc"}
    ),

    Document(
        page_content="Parrots are intelligent birds that can learn to imitate human speech.",
        metadata={"source": "birds-pets-doc"}
    ),

    Document(
        page_content="Rabbits are social and gentle animals that require a safe and comfortable environment.",
        metadata={"source": "small-pets-doc"}
    ),

    Document(
        page_content="Hamsters are small pets that are active mostly during the evening and night.",
        metadata={"source": "small-pets-doc"}
    )
]

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")

os.environ['LANGCHAIN_API_KEY']=os.getenv("LANGCHAIN_API_KEY")


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=50,chunk_overlap=25)

splitted_text=text_splitter.split_documents(documents)

In [4]:
splitted_text

[Document(metadata={'source': 'pets-doc'}, page_content='Dogs are loyal and friendly animals that make'),
 Document(metadata={'source': 'pets-doc'}, page_content='animals that make great companions.'),
 Document(metadata={'source': 'pets-doc'}, page_content='Cats are independent pets that enjoy spending time'),
 Document(metadata={'source': 'pets-doc'}, page_content='that enjoy spending time in their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are easy-to-care-for pets and are'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='pets and are suitable for beginners.'),
 Document(metadata={'source': 'birds-pets-doc'}, page_content='Parrots are intelligent birds that can learn to'),
 Document(metadata={'source': 'birds-pets-doc'}, page_content='birds that can learn to imitate human speech.'),
 Document(metadata={'source': 'small-pets-doc'}, page_content='Rabbits are social and gentle animals that require'),
 Document(metadata={'source':

In [5]:
# embeddignns

from langchain_ollama import OllamaEmbeddings

embeddings=OllamaEmbeddings(
    model='nomic-embed-text'
)

In [6]:
from langchain_chroma import Chroma

vector_store=Chroma.from_documents(splitted_text,embedding=embeddings)

vector_store

In [7]:
vector_store.similarity_search_with_score('cat')

[(Document(id='d322c892-0b9b-48d6-9b0c-61b12ac2fbdc', metadata={'source': 'pets-doc'}, page_content='Cats are independent pets that enjoy spending time'),
  0.5086807012557983),
 (Document(id='2b417aae-ca46-4780-8c9a-30666755d550', metadata={'source': 'small-pets-doc'}, page_content='Rabbits are social and gentle animals that require'),
  0.912516176700592),
 (Document(id='f79af6f9-fb22-49e1-8750-a9cd7550aa8e', metadata={'source': 'small-pets-doc'}, page_content='Hamsters are small pets that are active mostly'),
  0.9250794053077698),
 (Document(id='c8cb3d0b-1cb2-4f2f-9443-cf8ceb88dd9b', metadata={'source': 'pets-doc'}, page_content='animals that make great companions.'),
  0.9616100788116455)]

In [8]:
retriever=vector_store.as_retriever() # by default it is using search_type='similarity' menas based on vectors similarity ; mena retireives the documents whose vector is similar

# retriever = vector_store.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={
#         "score_threshold": 0.7
#     }
# ) ->only rerieve those documents whoise similarity scores meet the threshold

In [9]:
#llm
from langchain_groq import ChatGroq

llm=ChatGroq(
    model='openai/gpt-oss-120b',
    temperature=0
)

In [10]:
llm.invoke("hello")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "hello". We need to respond appropriately. It\'s a simple greeting. We can respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 72, 'total_tokens': 113, 'completion_time': 0.085384308, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.073253649, 'prompt_tokens_details': None, 'queue_time': 0.459541384, 'total_time': 0.158637957}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_87b3c396db', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a10059-3047-7713-9bc1-e609cceabfe5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 41, 'total_tokens': 113, 'output_token_details': {'reasoning': 23}})

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain


prompt=ChatPromptTemplate.from_messages([
    ("system","""Answer the question using the provided context only 
        context:{context}"""),
    ("user","{input}")

])

document_chain=create_stuff_documents_chain(llm,prompt)
retrival_chain=create_retrieval_chain(retriever,document_chain)



In [18]:
response = retrival_chain.invoke({
    "input": "tell me about cats?"
})

print(response["answer"])

Cats are independent pets that enjoy spending time and are suitable for beginners.
